# Contested Norms Study
## Data Prep: Create derived fields

Created: 2025-10-13  
Updated: 2025-10-20  
Authors: Edward L. Platt  

In [ ]:
%matplotlib inline
from configparser import ConfigParser
import csv
from datetime import datetime, timedelta
import json
import logging
import math
import os
import pytz
import simplejson as json
import sys
from tqdm import tqdm

utc=pytz.UTC

### LOAD CONFIGURATION
config = ConfigParser()
config.read('config.ini')
config.write(sys.stdout)

subreddit_id = config.get("Source", "subreddit_id")
start_time = utc.localize(datetime.strptime(config.get("Source", "start_time"), "%Y-%m-%d %H:%M:%S"))
end_time = utc.localize(datetime.strptime(config.get("Source", "end_time"), "%Y-%m-%d %H:%M:%S"))
post_file = config.get("Extracted", "post_file")
comment_file = config.get("Extracted", "comment_file")
modaction_file = config.get("Extracted", "modaction_file")
praw_post_file = config.get("Extracted", "praw_post_file")
praw_comment_file = config.get("Extracted", "praw_comment_file")

script_name = "contested_norms-{}-transform".format(subreddit_id)
script_date = datetime.now().strftime('%Y-%m-%d')

# Configure logging
logging.basicConfig(
    filename='{}-{}.log'.format(script_date, script_name),
    format='%(asctime)s:%(levelname)s:%(message)s',
    level=logging.DEBUG)
logger = logging.getLogger("CivilServant-Analysis")
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.DEBUG)
formatter = logging.Formatter('%(asctime)s:%(levelname)s:%(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

### LOAD ANALYSIS CODE
from pipeline import DataFile
from pipeline.sources import (
    CivilServantPostSource,
    CivilServantCommentSource,
    CivilServantModActionSource,
    PRAWPostSource,
    PRAWCommentSource)
from pipeline.transforms import CivilServantTransformToAccounts

In [ ]:
exclude = lambda row: row['author'] == 'AutoModerator'

post_source = CivilServantPostSource(subreddit_id, start_time, end_time)
posts = DataFile(
    post_source, filename=post_file, load=True, exclude=exclude)

comment_source = CivilServantCommentSource(subreddit_id, start_time, end_time)
comments = DataFile(
    comment_source, filename=comment_file, load=True, exclude=exclude)

modaction_source = CivilServantModActionSource(subreddit_id, start_time, end_time)
modactions = DataFile(modaction_source, filename=modaction_file, load=True)

praw_post_source = PRAWPostSource(subreddit_id, start_time, end_time)
praw_posts = DataFile(
    praw_post_source, filename=praw_post_file, load=True, exclude=exclude)

praw_comment_source = PRAWCommentSource(subreddit_id, start_time, end_time)
praw_comments = DataFile(
    praw_comment_source, filename=praw_comment_file, load=True, exclude=exclude)


In [ ]:
from importlib import reload
import pipeline
import pipeline.transforms
reload(pipeline)
reload(pipeline.transforms)
from pipeline.transforms import CivilServantTransformToAccounts

In [ ]:
accounts_source = CivilServantTransformToAccounts(
    start_time, end_time, subreddit_id, posts, comments, modactions, praw_posts, praw_comments)
accounts = DataFile(accounts_source)
accounts.transform(True)